# NeuroScope AI — 3D Glioma SegResNet Fine-Tuning on Google Colab (GPU)
**Project:** BrainAgenticthon 2026 — Medical AI Workstream (Rou'a & Abdulrahman)  
**Goal:** Fine-tune the trained 3D SegResNet model with `DiceCELoss` and high-resolution $96^3$ crops to eliminate false-positive background noise and cut HD95 to clinical levels.

---
### ⚡ Why Google Colab?
- **Local CPU:** 1 epoch takes ~38 minutes (20 epochs = ~27 hours).
- **Google Colab (T4 / A100 GPU):** 1 epoch takes **~35 to 45 seconds** (20 epochs = **~12 to 15 minutes**!).

## Step 1: Verify GPU Environment
Make sure your Colab runtime is set to **GPU**:  
`Runtime` ➔ `Change runtime type` ➔ Select **T4 GPU** (or **A100 GPU** if using Colab Pro).

In [ ]:
!nvidia-smi
import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:", torch.cuda.get_device_name(0))
else:
    print("WARNING: Running on CPU. Please switch to GPU runtime via Runtime -> Change runtime type!")

## Step 2: Clone GitHub Repository and Install Requirements

In [ ]:
import os
if not os.path.exists("MRI-"):
    !git clone https://github.com/M0maniiR0/MRI-.git
%cd MRI-
!pip install -q monai nibabel scipy

## Step 3: Download and Extract Task01_BrainTumour Dataset
Downloads the official MSD BraTS dataset (~2 GB) directly into Colab in under 1 minute.

In [ ]:
import os
dataset_dir = "Task01_BrainTumour/Task01_BrainTumour"
if not os.path.exists(f"{dataset_dir}/imagesTr"):
    print("Downloading Task01_BrainTumour archive from AWS S3 (~2 GB)...")
    !wget -q --show-progress https://msd-for-monai.s3-us-west-2.amazonaws.com/Task01_BrainTumour.tar
    print("Extracting archive...")
    !tar -xf Task01_BrainTumour.tar
    # Ensure nested directory structure matches repository expectations
    !mkdir -p Task01_BrainTumour/Task01_BrainTumour
    !cp -r Task01_BrainTumour/imagesTr Task01_BrainTumour/Task01_BrainTumour/ 2>/dev/null || true
    !cp -r Task01_BrainTumour/labelsTr Task01_BrainTumour/Task01_BrainTumour/ 2>/dev/null || true
    !cp -r Task01_BrainTumour/dataset.json Task01_BrainTumour/Task01_BrainTumour/ 2>/dev/null || true
    print("Dataset ready on disk!")
else:
    print("Dataset already exists on disk.")

## Step 4: Verify Root-Cause Discovery & Experiment A (Brain Mask Guard)
Test the Brain Mask Guard on the existing Epoch 20 baseline checkpoint.  
This immediately eliminates the 38-48% air leakage outside the skull, boosting WT Dice by **+13.5%** and cutting HD95 in half without retraining!

In [ ]:
!python diagnose_root_cause.py

## Step 5: Launch Phase 4 Fine-Tuning on GPU (12-15 Minutes Total)
Runs controlled fine-tuning starting from `models/segresnet_fullvolume_best.pt`:
- **Loss Function:** `DiceCELoss(sigmoid=True)` (adds Cross-Entropy to actively penalize background false positives)
- **Crop Size:** $96 \times 96 \times 96$ (broader anatomical context)
- **Learning Rate:** $5 \times 10^{-5}$ with Cosine Annealing scheduler
- **Output Checkpoint:** `models/segresnet_finetuned_best.pt`

In [ ]:
!python finetune_segresnet.py \
    --initial-checkpoint models/segresnet_fullvolume_best.pt \
    --output-checkpoint models/segresnet_finetuned_best.pt \
    --epochs 20 \
    --lr 5e-5 \
    --crop-size 96 96 96 \
    --batch-size 1 \
    --device cuda

## Step 6: Full Validation Evaluation of Fine-Tuned Model
Evaluates the new fine-tuned model across all 73 validation cases with the Brain Mask Guard.

In [ ]:
%cd /content/MRI-
!python segmentation_baseline.py \
    --evaluate \
    --checkpoint models/segresnet_finetuned_best.pt \
    --num-cases 73 \
    --output-dir outputs/segmentation_finetuned \
    --report-file reports/finetuned_evaluation_metrics.json \
    --device cuda


## Step 7: Generate Visual Comparison Figures & Overlay Montages

In [ ]:
%cd /content/MRI-
!python generate_segmentation_visualizations.py --output-dir outputs/segmentation_finetuned --report-file reports/finetuned_evaluation_metrics.json
from IPython.display import Image, display
import glob
vis_files = sorted(glob.glob('outputs/segmentation_finetuned/visualizations/*_comparison.png'))
if vis_files:
    display(Image(vis_files[0]))


## Step 8: Download Best Checkpoint & Push Results
Download `models/segresnet_finetuned_best.pt` directly to your local machine.

In [ ]:
%cd /content/MRI-
from google.colab import files
import os
if os.path.exists('models/segresnet_finetuned_best.pt'):
    files.download('models/segresnet_finetuned_best.pt')
    print('Downloaded models/segresnet_finetuned_best.pt successfully!')
else:
    print('Checkpoint not found. Run fine-tuning first.')
